<a href="https://colab.research.google.com/github/KohkiIsa/CoxPH-and-RSF-senior-thesis/blob/main/%E6%84%8F%E5%91%B3%E4%BF%9D%E6%8C%81%E5%BA%A6%E3%80%81%E6%A0%B9%E6%8B%A0%E4%BF%9D%E6%8C%81%E5%BA%A6%E3%80%81%E8%A1%A8%E7%8F%BE%E5%A4%9A%E6%A7%98%E5%BA%A6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade openai pyyaml

from openai import OpenAI
from getpass import getpass
import os
import time
import csv
import io
import random
import yaml
import re
import ast

# =========================
# 基本設定
# =========================

os.environ["OPENAI_API_KEY"] = getpass("APIキーを入力してください: ")
client = OpenAI()

MODEL = "gpt-5.4"
TEMPERATURE = 0.6
MAX_OUTPUT_TOKENS = 7000

YML_PATH = "/mnt/data/protocol_saved.yml"
if not os.path.exists(YML_PATH):
    YML_PATH = "protocol_saved.yml"

OUTPUT_FILE = "headache_emergency_calls_with_common_vitals_202606281200.csv"

RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

# 27個の頭痛ベクトルそれぞれ6件作る
# 3^3 * 6 = 162件
N_PER_HEADACHE_VECTOR = 6

# few-shotの入れ方を切り替える
# - "axis18": 3質問 × 3観点 × High/Low = 18通り
# - "combo24": 3質問 × 2^3 = 24通り
# - "none": few-shotなし
PROMPT_FEWSHOT_MODE = "axis18"

# =========================
# 頭痛ベクトル
# =========================

vectors = [
    [0,0,0,1],
    [0,0,1,1],
    [0,0,2,1],
    [0,1,0,1],
    [0,1,1,1],
    [0,1,2,1],
    [0,2,0,1],
    [0,2,1,1],
    [0,2,2,1],

    [1,0,0,1],
    [1,0,1,1],
    [1,0,2,1],
    [1,1,0,1],
    [1,1,1,2],
    [1,1,2,0],
    [1,2,0,0],
    [1,2,1,0],
    [1,2,2,0],

    [2,0,0,0],
    [2,0,1,0],
    [2,0,2,0],
    [2,1,0,0],
    [2,1,1,0],
    [2,1,2,0],
    [2,2,0,0],
    [2,2,1,0],
    [2,2,2,0],
]

def headache_label_from_d(d):
    return {0: "R3", 1: "R2", 2: "Y2"}[d]

def value_text(x):
    return {0: "はい", 1: "いいえ", 2: "不明"}[x]

# =========================
# YML読み込み
# =========================

with open(YML_PATH, "r", encoding="utf-8") as f:
    protocol = yaml.safe_load(f)

common_nodes = {
    node["id"]: node
    for node in protocol["common_vitals"]
    if "id" in node
}

COMMON_NODE_ORDER = [
    "common_breathing",
    "common_cold_sweat",
    "common_face_color",
    "common_conversation",
]

COMMON_NODE_LABEL = {
    "common_breathing": "呼吸",
    "common_cold_sweat": "冷や汗",
    "common_face_color": "顔色",
    "common_conversation": "会話可否",
}

# =========================
# 共通バイタルの数値対応
# =========================

COMMON_CODE_MAP = {}

for node_id in COMMON_NODE_ORDER:
    node = common_nodes[node_id]
    choices = node["choices"]
    COMMON_CODE_MAP[node_id] = {
        choice["code"]: idx
        for idx, choice in enumerate(choices)
    }

COMMON_INV_CODE_MAP = {}

for node_id in COMMON_NODE_ORDER:
    node = common_nodes[node_id]
    choices = node["choices"]
    COMMON_INV_CODE_MAP[node_id] = {
        idx: choice
        for idx, choice in enumerate(choices)
    }

# 頭痛症状質問へ進むための共通バイタル正常ルート
# 呼吸: a = はい
# 冷や汗: b = いいえ
# 顔色: b = 悪くない
# 会話可否: a = はい
ROUTE_TO_HEADACHE_COMMON_CODES = {
    "common_breathing": "a",
    "common_cold_sweat": "b",
    "common_face_color": "b",
    "common_conversation": "a",
}

ROUTE_TO_HEADACHE_VECTOR = [
    COMMON_CODE_MAP[node_id][ROUTE_TO_HEADACHE_COMMON_CODES[node_id]]
    for node_id in COMMON_NODE_ORDER
]

# =========================
# 共通バイタルのラベル説明
# =========================

COMMON_MEANING = {
    "common_breathing": {
        0: "呼吸は普段通り",
        1: "呼吸なし",
        2: "死戦期呼吸を疑う",
        3: "いびきのような呼吸",
        4: "窒息",
        5: "呼吸が苦しそう",
        6: "呼吸状態が不明",
    },
    "common_cold_sweat": {
        0: "冷や汗あり",
        1: "冷や汗なし",
        2: "冷や汗の有無が不明",
    },
    "common_face_color": {
        0: "顔色が悪い",
        1: "顔色は悪くない",
        2: "顔色が不明",
    },
    "common_conversation": {
        0: "普通に話せる",
        1: "つじつまが合わない",
        2: "単語しか話せない",
        3: "うめき声のみ",
        4: "声が全く出ない",
        5: "話の内容が不明",
        6: "呼吸苦で途切れ途切れにしか話せない",
        7: "会話可否が不明",
    },
}

# =========================
# 共通バイタル判定
# =========================

def choice_from_value(node_id, value):
    return COMMON_INV_CODE_MAP[node_id][value]

COMMON_INDEX_BY_NODE = {
    "common_breathing": 0,
    "common_cold_sweat": 1,
    "common_face_color": 2,
    "common_conversation": 3,
}

def trace_common_vitals_by_yml(common_vector):
    """
    共通バイタルをYMLのnextに従ってたどる。
    4項目すべてのベクトル値は保持するが、
    判定は「YML上で最初にtriageが出た地点」で止める。
    """

    node_id = "common_breathing"
    path = []

    while node_id in common_nodes:
        node = common_nodes[node_id]

        if node.get("metadata_only"):
            break

        if node_id not in COMMON_INDEX_BY_NODE:
            break

        idx = COMMON_INDEX_BY_NODE[node_id]
        value = common_vector[idx]
        choice = choice_from_value(node_id, value)

        step = {
            "node_id": node_id,
            "value": value,
            "code": choice.get("code"),
            "text": choice.get("text"),
            "triage": choice.get("triage", ""),
            "next": choice.get("next", ""),
        }
        path.append(step)

        if "triage" in choice:
            return {
                "common_path_triage": choice["triage"],
                "common_terminal_node": node_id,
                "common_terminal_value": value,
                "common_terminal_code": choice.get("code"),
                "common_terminal_text": choice.get("text"),
                "common_reaches_symptom_by_yml": False,
                "common_path": path,
            }

        next_id = choice.get("next")

        if next_id == "route_to_symptom_inquiry":
            return {
                "common_path_triage": "",
                "common_terminal_node": "route_to_symptom_inquiry",
                "common_terminal_value": "",
                "common_terminal_code": "",
                "common_terminal_text": "症状別質問へ進む",
                "common_reaches_symptom_by_yml": True,
                "common_path": path,
            }

        if not next_id:
            return {
                "common_path_triage": "",
                "common_terminal_node": node_id,
                "common_terminal_value": value,
                "common_terminal_code": choice.get("code"),
                "common_terminal_text": choice.get("text"),
                "common_reaches_symptom_by_yml": False,
                "common_path": path,
            }

        node_id = next_id

    raise RuntimeError(f"共通バイタル遷移が想定外の場所で終了しました: {node_id}")

# =========================
# 頭痛プロトコル取得
# =========================

headache_protocol = None

for p in protocol["protocols"]:
    if p.get("id") == "headache":
        headache_protocol = p
        break

if headache_protocol is None:
    raise ValueError("YML内に headache プロトコルが見つかりません。")

headache_nodes = {
    node["id"]: node
    for node in headache_protocol["nodes"]
    if "id" in node
}

HEADACHE_VALUE_TO_CODE = {
    0: "a",
    1: "b",
    2: "c",
}

HEADACHE_INDEX_BY_NODE = {
    "headache_sudden_severe": 0,
    "headache_numbness_paralysis": 1,
    "headache_abnormal_behavior": 2,
}

def choice_from_code_in_node(node, code):
    for choice in node.get("choices", []):
        if choice.get("code") == code:
            return choice
    raise ValueError(f"{node['id']} に code={code} が見つかりません。")

def trace_headache_by_yml(headache_vector):
    node_id = "headache_sudden_severe"
    path = []

    while node_id in headache_nodes:
        node = headache_nodes[node_id]

        if "choices" not in node or not node.get("choices"):
            break

        if node_id not in HEADACHE_INDEX_BY_NODE:
            break

        idx = HEADACHE_INDEX_BY_NODE[node_id]
        value = headache_vector[idx]
        code = HEADACHE_VALUE_TO_CODE[value]
        choice = choice_from_code_in_node(node, code)

        step = {
            "node_id": node_id,
            "value": value,
            "code": code,
            "text": choice.get("text"),
            "triage": choice.get("triage", ""),
            "next": choice.get("next", ""),
        }
        path.append(step)

        if "triage" in choice:
            return {
                "headache_path_triage": choice["triage"],
                "headache_terminal_node": node_id,
                "headache_terminal_value": value,
                "headache_terminal_code": code,
                "headache_terminal_text": choice.get("text"),
                "headache_path": path,
            }

        next_id = choice.get("next")
        if not next_id:
            break

        node_id = next_id

    a, b, c, d = headache_vector

    if 2 in [a, b, c]:
        fallback_label = headache_protocol["fallback"]["if_any_unknown"]
    elif a == 1 and b == 1 and c == 1:
        fallback_label = headache_protocol["fallback"]["if_all_symptom_questions_negative"]
    else:
        fallback_label = headache_label_from_d(d)

    return {
        "headache_path_triage": fallback_label,
        "headache_terminal_node": "headache_fallback",
        "headache_terminal_value": "",
        "headache_terminal_code": "",
        "headache_terminal_text": "fallback",
        "headache_path": path,
    }

# =========================
# 共通トリアージ10件の設計
# =========================

# 162件中10件だけ、共通項目でトリアージ可能にする。
# その10件のうち1件は「患者を直接見ることができない → R3」にする。
REQUIRED_COMMON_TRIAGE_VECTORS = [
    [6, 2, 2, 7],  # 患者を直接見ることができない → 呼吸状態不明でR3

    [1, 1, 0, 4],  # 呼吸なし → R1
    [2, 1, 2, 7],  # 死戦期呼吸疑い → R1
    [4, 0, 0, 4],  # 窒息 → R1
    [3, 0, 0, 3],  # いびきのような呼吸 → R2
    [5, 0, 0, 6],  # 呼吸が苦しそう → R2
    [0, 0, 0, 0],  # 冷や汗あり → R2
    [0, 2, 1, 0],  # 冷や汗の有無が不明 → R3
    [0, 1, 0, 1],  # 顔色が悪い → R2
    [0, 1, 1, 1],  # 会話のつじつまが合わない → R2
]

if len(REQUIRED_COMMON_TRIAGE_VECTORS) != 10:
    raise ValueError(f"共通トリアージケースが10件ではありません: {len(REQUIRED_COMMON_TRIAGE_VECTORS)}件")

# 「患者を直接見ることができない → R3」ケースが1件だけ入っているか確認
UNOBSERVABLE_R3_VECTOR = [6, 2, 2, 7]

unobservable_count = sum(
    1 for v in REQUIRED_COMMON_TRIAGE_VECTORS
    if v == UNOBSERVABLE_R3_VECTOR
)

if unobservable_count != 1:
    raise ValueError(
        "患者を直接見ることができない→R3 のケースが1件のみではありません。"
        f"実際: {unobservable_count}件"
    )

# 162件のうち、どの位置に共通トリアージケース10件を入れるか
# 1〜162の通し番号で指定
COMMON_TRIAGE_GLOBAL_POSITIONS = [1, 17, 33, 49, 65, 81, 97, 113, 129, 145]

COMMON_TRIAGE_VECTOR_BY_GLOBAL_POSITION = dict(
    zip(COMMON_TRIAGE_GLOBAL_POSITIONS, REQUIRED_COMMON_TRIAGE_VECTORS)
)

# それぞれ本当に共通バイタルでtriage確定するか確認
for global_pos, common_vector in COMMON_TRIAGE_VECTOR_BY_GLOBAL_POSITION.items():
    trace = trace_common_vitals_by_yml(common_vector)
    if not trace["common_path_triage"]:
        raise ValueError(
            f"共通トリアージケースがtriage確定しません。"
            f"global_pos={global_pos}, common_vector={common_vector}, trace={trace}"
        )

# =========================
# ケース作成
# =========================

def make_common_case(common_vector):
    common_trace = trace_common_vitals_by_yml(common_vector)

    special_note = ""
    if common_vector == UNOBSERVABLE_R3_VECTOR:
        special_note = "特殊ケース: 通報者が患者を直接見ることができず、呼吸・冷や汗・顔色・会話可否が不明のためR3"

    return {
        "common_vector": common_vector,

        "common_path_triage": common_trace["common_path_triage"],
        "common_terminal_node": common_trace["common_terminal_node"],
        "common_terminal_value": common_trace["common_terminal_value"],
        "common_terminal_code": common_trace["common_terminal_code"],
        "common_terminal_text": common_trace["common_terminal_text"],
        "common_reaches_symptom_by_yml": common_trace["common_reaches_symptom_by_yml"],
        "common_path": common_trace["common_path"],

        # 共通バイタルでトリアージ可能でも、主訴である頭痛を深掘りする
        "dataset_continue_to_headache": True,

        "special_note": special_note,
    }

def describe_common_case(case):
    lines = []

    if case.get("special_note"):
        lines.append(f"【特殊ケース】{case['special_note']}")

    lines.append("【共通バイタル全項目の観察値】")
    for node_id, value in zip(COMMON_NODE_ORDER, case["common_vector"]):
        label = COMMON_NODE_LABEL[node_id]
        meaning = COMMON_MEANING[node_id][value]
        choice = choice_from_value(node_id, value)
        triage = choice.get("triage", "")

        if triage:
            lines.append(f"- {label}: {value} = {meaning} / この項目単体では {triage}")
        else:
            lines.append(f"- {label}: {value} = {meaning}")

    lines.append("【YML遷移としての共通バイタル判定】")
    if case["common_path_triage"]:
        lines.append(
            f"- {case['common_terminal_node']} の "
            f"{case['common_terminal_value']}={case['common_terminal_text']} で "
            f"{case['common_path_triage']} と一旦判定"
        )
    else:
        lines.append("- 共通バイタルでは判定確定せず、YML上は症候別質問へ進む")

    lines.append("【データセット作成上の扱い】")
    lines.append("- この会話では、共通バイタル後も主訴である頭痛a,b,cを確認する")

    return "\n".join(lines)

def make_cases_for_vector(vector_index):
    """
    各頭痛ベクトルにつき6件作る。
    合計162件のうち、指定した10件だけ共通バイタルでtriage確定。
    それ以外は共通バイタル正常ルートで頭痛症状質問へ進む。
    """

    cases = []

    for j in range(1, N_PER_HEADACHE_VECTOR + 1):
        global_case_index = (vector_index - 1) * N_PER_HEADACHE_VECTOR + j

        if global_case_index in COMMON_TRIAGE_VECTOR_BY_GLOBAL_POSITION:
            common_vector = COMMON_TRIAGE_VECTOR_BY_GLOBAL_POSITION[global_case_index].copy()
        else:
            common_vector = ROUTE_TO_HEADACHE_VECTOR.copy()

        case = make_common_case(common_vector)
        case["id"] = f"v{vector_index:03d}-{j:02d}"
        cases.append(case)

    return cases

# =========================
# プロンプト
# =========================

developer_prompt = """
あなたは頭痛に関する119番救急電話会話データを作成するアシスタントです。

重要:
- CSV形式のテキストとして出力してください。
- JSON形式、Excel形式、Markdown表にはしないでください。
- 出力はCSVの行だけにしてください。
- ヘッダー行は出力しないでください。
- CSVの列は必ず以下の2列だけにしてください:
ID,会話文
- 1つの会話をCSVの1行にしてください。
- 各セルは必ずダブルクォーテーションで囲んでください。
- 会話文の中では実際の改行を使わず、発話の区切りは「\\n」という文字列で表してください。
- 会話文はDispatcherとCallerの発話を交互に含め、会話全体をまるまる載せてください。
- 患者本人ではなく、必ず第三者が通報している設定にしてください。
- 会話本文に R1 / R2 / R3 / Y2 という文字は絶対に出さないでください。
- 同じ構文ばかりにせず、通報者・患者・場所・言い回しを変えてください。
- 「はい、しびれはありません」のような矛盾文は禁止です。

共通バイタルについて:
- 呼吸、冷や汗、顔色、会話可否の4項目は、必ず会話文から判断できるようにしてください。
- 4項目すべてについて、Dispatcherが質問するか、Callerが明確に説明する形にしてください。
- 共通バイタルで重い状態が含まれる場合でも、救急車を向かわせながら追加確認する自然な会話にしてください。
- YML上、共通バイタルで一旦判定が確定する場合でも、このデータセットではその後に頭痛a,b,cを確認してください。
- 共通バイタルの4項目は、呼吸→冷や汗→顔色→会話可否の順に自然に確認してください。
- 頭痛症状では、a→b→cの順に自然に確認してください。
- 共通バイタルによる判定と、頭痛症状による判定はPython側で別々に計算します。会話文ではラベル判定を説明しないでください。
- Callerの初期説明として頭痛が主訴であることは必ず含めてください。
- 会話ラリー数は多めにしてください。住所、年齢、意識、呼吸、安全確保、救急車到着までの案内なども自然に含めてください。
"""

YML_EVIDENCE_RULES_TEXT = """
YML遷移図に基づく根拠保持度の基準:
- a: headache_sudden_severe
  - 値0: 突然の激しい頭痛がある。ここで頭痛症状としての判定が確定する。
  - 値1: 突然の激しい頭痛ではない。次の b に進む。
  - 値2: 突然の激しい頭痛か不明。不明を含むためfallback側の扱いになる。
- b: headache_numbness_paralysis
  - 値0: しびれや麻痺がある。ここで頭痛症状としての判定が確定する。
  - 値1: しびれや麻痺はない。次の c に進む。
  - 値2: しびれや麻痺が不明。不明を含むためfallback側の扱いになる。
- c: headache_abnormal_behavior
  - 値0: いつもと違う振る舞いがある。ここで頭痛症状としての判定が確定する。
  - 値1: いつもと違う振る舞いはない。a,b,cがすべて陰性ならfallback側の軽い判定になる。
  - 値2: いつもと違う振る舞いが不明。不明を含むためfallback側の扱いになる。

根拠保持度High:
- その1ラリーから、対象ノードの値0/1/2を復元できる。
- aなら「突然性」と「激しさ」、bなら「しびれ/麻痺」、cなら「普段と違う振る舞い」を判断できる。

根拠保持度Low:
- 頭痛や体調の話はしていても、対象ノードの値0/1/2を復元できない。
- 別ノードの根拠だけ答えている、または「様子が悪い」など抽象的すぎる回答はLowとする。
"""

AXIS18_FEWSHOT_EXAMPLES = {
    "a": {
        "label": "激しい痛みが突然起こったか",
        "examples": [
            ("意味保持度", "High", "YML値=2", "Dispatcher: 頭の痛みは、突然起こった激しい痛みですか？", "Caller: すみません、本人から頭痛とだけ連絡が来ていて、急に強くなったのかまでは分かりません。"),
            ("意味保持度", "Low", "YML値=復元不可", "Dispatcher: いつもの軽い頭痛ということでよいですか？", "Caller: そこまでは分かりません。"),
            ("根拠保持度", "High", "YML値=2", "Dispatcher: 頭の痛みは、急に出た強い痛みか確認できますか？", "Caller: 近くにいないので、突然なのか、強い痛みなのかは確認できません。"),
            ("根拠保持度", "Low", "YML値=復元不可", "Dispatcher: 頭の痛みについて何か分かることはありますか？", "Caller: 頭が痛いとだけ聞いています。"),
            ("表現多様度", "High", "YML値=2", "Dispatcher: さっきまで普通だったのに、急に我慢できないほど頭が痛くなった感じですか？", "Caller: 画面越しに連絡を受けただけなので、始まり方や痛みの強さまでは聞き取れていません。"),
            ("表現多様度", "Low", "YML値=2", "Dispatcher: 激しい痛みが突然起こりましたか？", "Caller: 激しい痛みが突然起こったかは分かりません。"),
        ],
    },
    "b": {
        "label": "しびれや麻痺はあるか",
        "examples": [
            ("意味保持度", "High", "YML値=2", "Dispatcher: 手足のしびれや麻痺はありますか？", "Caller: 今は本人のそばにいないので、しびれや麻痺があるかは確認できません。"),
            ("意味保持度", "Low", "YML値=復元不可", "Dispatcher: 普通に歩けているということでよいですか？", "Caller: そこは分かりません。"),
            ("根拠保持度", "High", "YML値=2", "Dispatcher: 片側の手足にしびれや動かしにくさがあるか見られますか？", "Caller: 別室にいて姿が見えないので、しびれや麻痺の有無は分かりません。"),
            ("根拠保持度", "Low", "YML値=復元不可", "Dispatcher: 体の様子で気になることはありますか？", "Caller: 頭が痛いと言っていること以外は聞けていません。"),
            ("表現多様度", "High", "YML値=2", "Dispatcher: 右手だけ力が入らない、足がしびれる、そういった様子は確認できますか？", "Caller: 電話口では痛いと言っているだけで、手足の感覚や動きまでは確認できていません。"),
            ("表現多様度", "Low", "YML値=2", "Dispatcher: しびれや麻痺はありますか？", "Caller: しびれや麻痺があるかは分かりません。"),
        ],
    },
    "c": {
        "label": "何かいつもと違う振る舞いはあるか",
        "examples": [
            ("意味保持度", "High", "YML値=2", "Dispatcher: 普段と違う振る舞いや様子はありますか？", "Caller: 今日はまだ直接会えていないので、いつもと違う様子かどうかは分かりません。"),
            ("意味保持度", "Low", "YML値=復元不可", "Dispatcher: いつもより元気で普通に過ごしていますか？", "Caller: そこは分かりません。"),
            ("根拠保持度", "High", "YML値=2", "Dispatcher: ぼんやりしている、返事がおかしいなど、普段と違う様子は見えますか？", "Caller: 今は電話で聞いただけなので、普段と違う振る舞いがあるかまでは確認できません。"),
            ("根拠保持度", "Low", "YML値=復元不可", "Dispatcher: 何か変わったことはありますか？", "Caller: 頭が痛いこと以外はまだ分かりません。"),
            ("表現多様度", "High", "YML値=2", "Dispatcher: 受け答えが変、落ち着きがない、いつもと違う行動をしている感じはありますか？", "Caller: 家族から頭痛とだけ聞いていて、普段との違いまではまだ見られていません。"),
            ("表現多様度", "Low", "YML値=2", "Dispatcher: いつもと違う振る舞いはありますか？", "Caller: いつもと違う振る舞いがあるかは分かりません。"),
        ],
    },
}

COMBO24_FEWSHOT_EXAMPLES = {
    "a": {
        "label": "激しい痛みが突然起こったか",
        "examples": [
            ("High", "High", "High", "YML値=0", "Dispatcher: さっきまで普通だったのに、急に我慢できないほど頭が痛くなった感じですか？", "Caller: はい、急に頭が割れるように痛いと言い出しました。"),
            ("High", "High", "Low", "YML値=0", "Dispatcher: 激しい痛みが突然起こりましたか？", "Caller: はい、激しい痛みが突然起こりました。"),
            ("High", "Low", "High", "YML値=復元不可", "Dispatcher: 頭痛の出方や強さについて、どんなふうに聞いていますか？", "Caller: 頭が痛いという連絡だけで、詳しい言い方までは聞けていません。"),
            ("High", "Low", "Low", "YML値=復元不可", "Dispatcher: 激しい痛みが突然起こりましたか？", "Caller: 頭痛です。"),
            ("Low", "High", "High", "YML値=0", "Dispatcher: いつもの片頭痛の続きという理解でよいですか？", "Caller: いいえ、今回はさっき急に、今までにないくらい強く痛くなったそうです。"),
            ("Low", "High", "Low", "YML値=1", "Dispatcher: いつもの頭痛ですか？", "Caller: はい、突然の激しい痛みではありません。"),
            ("Low", "Low", "High", "YML値=復元不可", "Dispatcher: 体調全体としては、どんな困り方をしていますか？", "Caller: 苦しそうで、頭のことしか言わないと聞いています。"),
            ("Low", "Low", "Low", "YML値=復元不可", "Dispatcher: 大丈夫そうですか？", "Caller: よく分かりません。"),
        ],
    },
    "b": {
        "label": "しびれや麻痺はあるか",
        "examples": [
            ("High", "High", "High", "YML値=0", "Dispatcher: 片側の手足にしびれがある、力が入らないといった様子はありますか？", "Caller: はい、右手がしびれて、うまく動かせないと言っています。"),
            ("High", "High", "Low", "YML値=0", "Dispatcher: しびれや麻痺はありますか？", "Caller: はい、しびれや麻痺があります。"),
            ("High", "Low", "High", "YML値=復元不可", "Dispatcher: 手足の感覚や動きで気になることは確認できますか？", "Caller: 頭が痛いとしか聞いておらず、体の動きの話はまだ聞けていません。"),
            ("High", "Low", "Low", "YML値=復元不可", "Dispatcher: しびれや麻痺はありますか？", "Caller: 体調が悪そうです。"),
            ("Low", "High", "High", "YML値=0", "Dispatcher: 転んだりぶつけたりした様子はありますか？", "Caller: 転倒ではないですが、左腕に力が入らないと言っています。"),
            ("Low", "High", "Low", "YML値=1", "Dispatcher: 手は大丈夫ですか？", "Caller: はい、しびれや麻痺はありません。"),
            ("Low", "Low", "High", "YML値=復元不可", "Dispatcher: 他に変なところはありますか？", "Caller: 何となく動きづらそうとは聞いていますが、詳しい部位は分かりません。"),
            ("Low", "Low", "Low", "YML値=復元不可", "Dispatcher: 大丈夫そうですか？", "Caller: よく分かりません。"),
        ],
    },
    "c": {
        "label": "何かいつもと違う振る舞いはあるか",
        "examples": [
            ("High", "High", "High", "YML値=0", "Dispatcher: 受け答えがかみ合わない、ぼんやりしているなど、普段と違う様子はありますか？", "Caller: はい、返事の内容がいつもと違って、話がかみ合いません。"),
            ("High", "High", "Low", "YML値=0", "Dispatcher: いつもと違う振る舞いはありますか？", "Caller: はい、いつもと違う振る舞いがあります。"),
            ("High", "Low", "High", "YML値=復元不可", "Dispatcher: 普段と違う行動や受け答えは見られますか？", "Caller: 頭痛の連絡だけで、今の様子までは見られていません。"),
            ("High", "Low", "Low", "YML値=復元不可", "Dispatcher: いつもと違う振る舞いはありますか？", "Caller: 頭痛です。"),
            ("Low", "High", "High", "YML値=0", "Dispatcher: 眠っていて起こせない状態ですか？", "Caller: 眠ってはいませんが、意味の通らない返事をしていて普段と違います。"),
            ("Low", "High", "Low", "YML値=1", "Dispatcher: 普通ですか？", "Caller: はい、いつもと違う振る舞いはありません。"),
            ("Low", "Low", "High", "YML値=復元不可", "Dispatcher: 何か困っていることはありますか？", "Caller: 家の中が慌ただしいとは聞いていますが、本人の様子は分かりません。"),
            ("Low", "Low", "Low", "YML値=復元不可", "Dispatcher: 大丈夫そうですか？", "Caller: よく分かりません。"),
        ],
    },
}

def build_axis18_fewshot_section():
    lines = [
        "不明回答のfew-shot例（18通り）:",
        "- 目的: a,b,cが「不明」のときの自然な1ラリー表現を増やすための例です。",
        "- 注意: 18例すべて、Callerの返答は対象質問に対して「不明」です。",
        "- 注意: Low例は品質目標ではなく、各観点の違いを示す分類例です。",
        "- 実際の生成では、指定された頭痛ベクトルの値を最優先し、意味・根拠を勝手に変えないでください。",
        YML_EVIDENCE_RULES_TEXT.strip(),
    ]

    for q_key, spec in AXIS18_FEWSHOT_EXAMPLES.items():
        lines.append(f"\n[{q_key}: {spec['label']}]")
        for axis, level, yml_value, question, answer in spec["examples"]:
            lines.append(f"- {axis}={level} / 返答=不明 / {yml_value}")
            lines.append(f"  {question}")
            lines.append(f"  {answer}")

    return "\n".join(lines)

def build_combo24_fewshot_section():
    lines = [
        "症状別質問のfew-shot例（24通り）:",
        "- 目的: a,b,cそれぞれについて、意味保持度・根拠保持度・表現多様度のHigh/Low組み合わせ別に1ラリー表現を示すための例です。",
        "- 注意: この24例では、Callerの返答は「はい」「いいえ」「不明」のいずれでも構いません。",
        "- 注意: Lowを含む例は品質目標ではなく、観点差を示す分類例です。",
        "- 実際の生成では、指定された頭痛ベクトルの値を最優先し、意味・根拠を勝手に変えないでください。",
        YML_EVIDENCE_RULES_TEXT.strip(),
    ]

    for q_key, spec in COMBO24_FEWSHOT_EXAMPLES.items():
        lines.append(f"\n[{q_key}: {spec['label']}]")
        for meaning_level, evidence_level, diversity_level, yml_value, question, answer in spec["examples"]:
            lines.append(
                "- "
                f"意味保持度={meaning_level}, "
                f"根拠保持度={evidence_level}, "
                f"表現多様度={diversity_level} / {yml_value}"
            )
            lines.append(f"  {question}")
            lines.append(f"  {answer}")

    return "\n".join(lines)

def build_fewshot_section(mode):
    if mode == "axis18":
        return build_axis18_fewshot_section()
    if mode == "combo24":
        return build_combo24_fewshot_section()
    if mode == "none":
        return "不明回答のfew-shot例: 今回は使用しない。"
    raise ValueError(f"未知のPROMPT_FEWSHOT_MODEです: {mode}")

def make_prompt(vector_index, vector, cases, fewshot_mode=PROMPT_FEWSHOT_MODE):
    n_cases = len(cases)
    a, b, c, d = vector
    headache_triage_label = headache_label_from_d(d)
    fewshot_section = build_fewshot_section(fewshot_mode)

    case_blocks = []
    for case in cases:
        case_blocks.append(f"""
ID: {case["id"]}

共通バイタル条件:
{describe_common_case(case)}

共通バイタル全項目ベクトル:
{case["common_vector"]}

YML上の共通バイタル遷移判定:
{case["common_path_triage"] if case["common_path_triage"] else "共通バイタルでは判定確定せず、症候別質問へ進む"}

YML上の共通バイタル停止ノード:
{case["common_terminal_node"]}

YML上、症状質問へ進むか:
{"進む" if case["common_reaches_symptom_by_yml"] else "進まない"}

データセット作成上、頭痛a,b,cを確認するか:
{"確認する" if case["dataset_continue_to_headache"] else "確認しない"}

頭痛条件:
a = 激しい痛みが突然起こったか: {a}（{value_text(a)}）
b = しびれや麻痺はあるか: {b}（{value_text(b)}）
c = 何かいつもと違う振る舞いはあるか: {c}（{value_text(c)}）
頭痛だけで判定した場合のd = {d}（{headache_triage_label}）
""")

    return f"""
以下の{n_cases}件について、頭痛に関する119番救急電話会話を作成してください。

vector_index: {vector_index}
頭痛ベクトル: {vector}

頭痛条件:
a = 激しい痛みが突然起こったか
今回の値: {a}（{value_text(a)}）

b = しびれや麻痺はあるか
今回の値: {b}（{value_text(b)}）

c = 何かいつもと違う振る舞いはあるか
今回の値: {c}（{value_text(c)}）

d = 頭痛症状だけで判定した緊急度
今回の値: {d}（{headache_triage_label}）

共通バイタルの確認ルール:
- 呼吸、冷や汗、顔色、会話可否の4つは、必ず会話文から判断できるようにしてください。
- 共通バイタルベクトルは [呼吸, 冷や汗, 顔色, 会話可否] の順です。
- -1、未確認、不明扱いの未質問は作らないでください。
- 「不明」という値が指定されている場合は、Callerが「近くで見えない」「確認できない」「今は分からない」などと答える形にしてください。
- 特殊ケースに「通報者が患者を直接見ることができない」と書かれている場合は、通報者が患者の近くにいない、別室・屋外・電話越しで直接見えない、暗くて確認できないなど、患者を直接観察できない状況として自然に表現してください。
- YML上、共通バイタルでR1/R2/R3が一旦確定する場合でも、このデータセットではその後に頭痛a,b,cも確認してください。
- ただし、会話文の中に R1 / R2 / R3 / Y2 というラベル名は出さないでください。
- 共通バイタルの4項目は、呼吸→冷や汗→顔色→会話可否の順に自然に確認してください。
- 頭痛症状では、a→b→cの順に自然に確認してください。
- 共通バイタルによる判定と、頭痛症状による判定は、Python側で別々に計算します。会話文ではラベル判定を説明しないでください。
- 主訴として頭痛があることは必ず会話の最初に入れてください。

頭痛a,b,cの意味:
- a=0なら、突然の激しい頭痛がある。
- a=1なら、突然の激しい頭痛ではない。徐々に痛くなった、前から続いている等。
- a=2なら、突然かどうか不明。
- b=0なら、しびれや麻痺がある。
- b=1なら、しびれや麻痺はない。
- b=2なら、しびれや麻痺は不明。
- c=0なら、いつもと違う振る舞いがある。
- c=1なら、いつもと違う振る舞いはない。
- c=2なら、いつもと違う振る舞いは不明。

few-shot設定:
{fewshot_section}

IDごとの条件:
{chr(10).join(case_blocks)}

会話の長さ:
- 1会話あたり、DispatcherとCallerの発話を合わせて20〜24発話程度にしてください。
- 短すぎる会話は禁止です。
- 最低限、以下を自然に含めてください。
  - 救急か火事かの確認
  - どうしたか
  - 場所
  - 患者の年齢や属性
  - 呼吸
  - 冷や汗
  - 顔色
  - 会話可否
  - 頭痛a,b,c
  - 救急車到着までの指示

出力形式:
CSV形式で出力してください。
Markdownの ```csv は付けないでください。
ヘッダー行は付けないでください。
データ行のみ{n_cases}行出力してください。

列順:
ID,会話文

CSVの注意:
- 各セルは必ずダブルクォーテーションで囲んでください。
- 会話文の中では実際の改行を使わず、\\n という文字列で発話を区切ってください。
- 余計な説明文は出力しないでください。
- 会話文セルの中には、DispatcherとCallerの発話をすべて入れてください。

生成を開始してください。
"""

# =========================
# CSV生成
# =========================

CSV_HEADER = [
    "ID",
    "会話文",

    "共通_呼吸",
    "共通_冷や汗",
    "共通_顔色",
    "共通_会話可否",
    "共通バイタル全項目ベクトル",

    "共通バイタル遷移判定",
    "共通バイタル停止ノード",
    "共通バイタル停止値",
    "共通バイタル停止コード",
    "共通バイタル停止内容",
    "YML上_症状質問へ進むか",

    "データ生成上_頭痛質問を続行したか",

    "激しい頭痛",
    "痺れ麻痺",
    "いつもと違う",
    "頭痛だけで判定した緊急度",

    "頭痛症状遷移判定",
    "頭痛症状停止ノード",
    "頭痛症状停止値",
    "頭痛症状停止コード",
    "頭痛症状停止内容",

    "最終総合緊急度ラベル",
]

all_rows = [CSV_HEADER]

def clean_model_text(text):
    return (
        text.strip()
        .replace("```csv", "")
        .replace("```", "")
        .strip()
    )

def parse_two_column_csv(text):
    """
    モデル出力を ID,会話文 の2列として読む。
    通常のCSVとして読める場合はcsv.readerを使う。
    行同士がくっついた場合は、IDパターンで強制的に切り出す。
    """

    try:
        reader = csv.reader(io.StringIO(text))
        raw_rows = list(reader)

        parsed = []

        for row in raw_rows:
            if len(row) == 0:
                continue

            if len(row) == 2:
                parsed.append([row[0], row[1]])
                continue

            if len(row) > 2 and len(row) % 2 == 0:
                for k in range(0, len(row), 2):
                    parsed.append([row[k], row[k + 1]])
                continue

            raise ValueError

        return parsed

    except Exception:
        pass

    id_pattern = re.compile(r'"?(v\d{3}-\d{2})"?\s*,\s*"')
    matches = list(id_pattern.finditer(text))

    if not matches:
        raise ValueError(f"モデル出力からIDを検出できませんでした: {text[:300]}")

    parsed = []

    for idx, match in enumerate(matches):
        row_id = match.group(1)
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)

        conversation = text[start:end].strip()
        conversation = conversation.rstrip()
        conversation = conversation.rstrip('", \n\r\t')
        conversation = conversation.replace('""', '"')

        parsed.append([row_id, conversation])

    return parsed

def final_label_from_common_and_headache(common_label, headache_label):
    """
    共通バイタルでtriageが出ているなら、それを最優先で採用。
    共通バイタルで判定が出ていない場合だけ、頭痛症状の判定を採用。
    """

    if common_label:
        return common_label

    if headache_label:
        return headache_label

    return ""

def append_generated_rows(vector, cases, rows):
    a, b, c, d = vector

    row_by_id = {row[0]: row[1] for row in rows}

    for case in cases:
        expected_id = case["id"]

        if expected_id not in row_by_id:
            raise ValueError(f"{expected_id} がモデル出力にありません。")

        conversation = row_by_id[expected_id].replace("\\n", "\n")

        common_vector = case["common_vector"]
        headache_trace = trace_headache_by_yml(vector)

        common_label = case["common_path_triage"]
        headache_label_by_yml = headache_trace["headache_path_triage"]

        final_label = final_label_from_common_and_headache(
            common_label,
            headache_label_by_yml
        )

        fixed_row = [
            expected_id,
            conversation,

            str(common_vector[0]),
            str(common_vector[1]),
            str(common_vector[2]),
            str(common_vector[3]),
            str(common_vector),

            common_label,
            case["common_terminal_node"],
            str(case["common_terminal_value"]),
            str(case["common_terminal_code"]),
            str(case["common_terminal_text"]),
            "1" if case["common_reaches_symptom_by_yml"] else "0",

            "1" if case["dataset_continue_to_headache"] else "0",

            str(a),
            str(b),
            str(c),
            str(d),

            headache_label_by_yml,
            headache_trace["headache_terminal_node"],
            str(headache_trace["headache_terminal_value"]),
            str(headache_trace["headache_terminal_code"]),
            str(headache_trace["headache_terminal_text"]),

            final_label,
        ]

        all_rows.append(fixed_row)

# =========================
# 生成実行：合計162件
# =========================

for i, vector in enumerate(vectors, start=1):
    cases = make_cases_for_vector(i)

    prompt = make_prompt(i, vector, cases)

    response = client.responses.create(
        model=MODEL,
        instructions=developer_prompt,
        input=prompt,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )

    text = clean_model_text(response.output_text)

    print("=" * 100)
    print(f"vector_index: {i}")
    print(f"headache_vector: {vector}")
    print(text)

    rows = parse_two_column_csv(text)

    expected_ids = {case["id"] for case in cases}
    rows = [row for row in rows if row[0] in expected_ids]

    if len(rows) != len(cases):
        raise ValueError(
            f"vector_index {i} の出力行数が{len(cases)}行ではありません。"
            f"実際: {len(rows)}行"
        )

    append_generated_rows(vector, cases, rows)

    time.sleep(0.5)

# =========================
# CSV保存
# =========================

with open(OUTPUT_FILE, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerows(all_rows)

print("保存完了:", OUTPUT_FILE)
print("保存行数:", len(all_rows) - 1)

# =========================
# 簡易チェック
# =========================

import pandas as pd

df = pd.read_csv(OUTPUT_FILE)

print("shape:")
print(df.shape)

print("総件数:")
print(len(df))

print("YML上_症状質問へ進むか:")
print(df["YML上_症状質問へ進むか"].value_counts(dropna=False))

print("データ生成上_頭痛質問を続行したか:")
print(df["データ生成上_頭痛質問を続行したか"].value_counts(dropna=False))

print("共通バイタル遷移判定:")
print(df["共通バイタル遷移判定"].value_counts(dropna=False))

print("頭痛症状遷移判定:")
print(df["頭痛症状遷移判定"].value_counts(dropna=False))

print("最終総合緊急度ラベル:")
print(df["最終総合緊急度ラベル"].value_counts(dropna=False))

print("共通バイタルに -1 が含まれていないか:")
common_cols = ["共通_呼吸", "共通_冷や汗", "共通_顔色", "共通_会話可否"]
for col in common_cols:
    print(col, (df[col] == -1).sum())

common_triage_mask = (
    df["共通バイタル遷移判定"].notna()
    & (df["共通バイタル遷移判定"].astype(str).str.strip() != "")
)

print("共通バイタルでトリアージ確定した件数:")
print(common_triage_mask.value_counts())

print("共通バイタル遷移判定の内訳:")
print(df.loc[common_triage_mask, "共通バイタル遷移判定"].value_counts(dropna=False))

df["_common_vector_tuple"] = df["共通バイタル全項目ベクトル"].apply(
    lambda x: tuple(ast.literal_eval(x))
)

unobservable_mask = df["_common_vector_tuple"] == tuple(UNOBSERVABLE_R3_VECTOR)

print("患者を直接見ることができない→R3 の件数:")
print(unobservable_mask.sum())

if unobservable_mask.sum() != 1:
    raise ValueError(
        "患者を直接見ることができない→R3 のケースが1件のみではありません。"
        f"実際: {unobservable_mask.sum()}件"
    )

display(
    df.loc[
        unobservable_mask,
        [
            "ID",
            "共通バイタル全項目ベクトル",
            "共通バイタル遷移判定",
            "共通バイタル停止ノード",
            "共通バイタル停止内容",
            "頭痛症状遷移判定",
            "最終総合緊急度ラベル",
        ],
    ]
)

print("共通トリアージ確定ケース一覧:")
display(
    df.loc[
        common_triage_mask,
        [
            "ID",
            "共通バイタル全項目ベクトル",
            "共通バイタル遷移判定",
            "共通バイタル停止ノード",
            "共通バイタル停止内容",
            "頭痛症状遷移判定",
            "最終総合緊急度ラベル",
        ],
    ]
)

print("共通バイタルで判定せず、頭痛症状質問へ進んだ件数:")
route_mask = df["YML上_症状質問へ進むか"] == 1
print(route_mask.value_counts())

print("最初の数行:")
display(df.head())

# 一時チェック用列を消して再保存
df = df.drop(columns=["_common_vector_tuple"])
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("チェック列を除いて再保存しました:", OUTPUT_FILE)